# Projeto de Inteligência Artificial — Malhas Aéreas

Notebook de entrega final. Ele reúne, em uma única execução, as três fases do projeto:

| Fase | O que faz | Onde está no código |
|---|---|---|
| 1. Formulação e busca | Malha aérea como grafo; rota com A* (distância) | `src/environment`, `src/search` |
| 2. Dados e aprendizado | Dataset sintético, modelo de custo (tempo de voo) e A* com custo aprendido | `src/environment/dataset.py`, `src/environment/model.py` |
| 3. Agente com LLM | LLM interpreta o pedido e chama a ferramenta `buscar_rota` (nunca calcula rota sozinho) | `src/agent` |

**Como executar:** *Ambiente de execução → Executar tudo*. O notebook funciona **sem chave de LLM**: nesse caso, só a demonstração ao vivo do agente (seção 5) é substituída pela chamada direta à ferramenta.

## 0. Preparação

Clona o repositório (se ainda não existir) e instala as dependências. A célula pode ser executada mais de uma vez na mesma sessão.

In [ ]:
%cd /content
REPO = "https://github.com/soficb14/projeto-ia-malhas-aereas.git"
![ -d projeto-ia-malhas-aereas ] || git clone {REPO}
%cd /content/projeto-ia-malhas-aereas
!pip install -q -r requirements.txt

## 1. Configuração do LLM

O agente (Fase 3) usa um LLM por uma API compatível com OpenAI. A configuração vem de três *Secrets* do Colab (ícone de chave na barra lateral), **nunca escritos no código**:

| Secret | Conteúdo | Valor usado nos resultados |
|---|---|---|
| `LLM_API_KEY` | chave do provedor | (privada) |
| `LLM_BASE_URL` | URL da API do provedor | **PREENCHER: URL do provedor** |
| `LLM_MODEL` | ID exato do modelo | **PREENCHER: ID exato do modelo** |

Se algum Secret não existir (por exemplo, em outra conta Google), a célula abaixo avisa quais faltam e o notebook segue sem o LLM.

In [ ]:
import os

NOMES_LLM = ("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL")
try:
    from google.colab import userdata
    for nome in NOMES_LLM:
        if os.environ.get(nome):
            continue
        try:
            valor = userdata.get(nome)
        except Exception:          # Secret inexistente ou sem acesso do notebook
            valor = None
        if valor:
            os.environ[nome] = valor
except ImportError:                # fora do Colab: usa as variáveis de ambiente
    pass

faltando = [n for n in NOMES_LLM if not os.environ.get(n)]
TEM_LLM = not faltando
if TEM_LLM:
    print("LLM configurado.")
else:
    print("Secrets ausentes:", ", ".join(faltando))
    print("A demonstração do agente (seção 5) usará a ferramenta diretamente, sem LLM.")

## 2. Fase 1 — Busca com A*

Rota de menor distância entre ABQ e STL no mapa médio (25 aeroportos). A heurística é a distância em linha reta, que nunca superestima o custo real.

In [ ]:
from src.environment.loader import load_network
from src.search.astar import a_star_search

network = load_network("data/maps/mapa_medio.json", "data/maps/airports.json")
r = a_star_search(network, "ABQ", "STL", metric="distance")
print(" -> ".join(a.code for a in r.path), f"| {r.cost:.0f} km | nós expandidos: {r.expanded_nodes}")

## 3. Fase 2 — Dataset e modelo de custo

Gera um dataset sintético de voos (com valores faltantes em clima e vento), compara um baseline (média) com Regressão Linear e Random Forest por validação cruzada e avalia no conjunto de teste. Em seguida, mostra o erro absoluto médio por variável.

In [ ]:
from src.environment.dataset import generate_synthetic_flight_dataset
from src.environment.model import train_and_evaluate, error_analysis

df = generate_synthetic_flight_dataset(network, samples_per_flight=20)
print(df.isna().sum()[lambda s: s > 0])
best_pipeline, results, holdout = train_and_evaluate(df)
display(results.round(2))
for col, tabela in error_analysis(*holdout).items():
    display(tabela)

## 4. Custo aprendido na busca

O modelo da Fase 2 estima o tempo de cada voo, e o A* passa a minimizar **minutos** em vez de km. Comparamos o cenário normal com uma tempestade no trecho DEN → ORD: a rota deve desviar.

In [ ]:
from src.environment.model import build_cost_map
from src.search.heuristics import MAX_SPEED_KMH

bom = {"aircraft_type": "Narrow-body", "route_congestion_index": 1,
       "weather_condition": "Clear", "wind_component": 0.0, "time_of_day": "Off-peak"}
evento = {("DEN", "ORD"): {"weather_condition": "Storm", "route_congestion_index": 5,
                           "time_of_day": "Peak", "wind_component": -25.0}}

for nome, ov in (("Sem evento", None), ("Tempestade em DEN->ORD", evento)):
    network.set_learned_costs(build_cost_map(network, best_pipeline, bom,
                                             overrides=ov, max_speed_km_h=MAX_SPEED_KMH))
    r = a_star_search(network, "ABQ", "STL", metric="predicted_time")
    print(f"{nome}: {' -> '.join(a.code for a in r.path)} | {r.cost:.0f} min")
network.clear_learned_costs()

## 5. Fase 3 — Agente com LLM

O LLM só interpreta o pedido em linguagem natural e escolhe os argumentos da ferramenta `buscar_rota`; a rota e o tempo vêm sempre do A* com o custo aprendido.

- **Com LLM configurado** (seção 1): o agente responde, e mostramos latência, tokens e ferramentas chamadas. Se a API devolver erro, ele aparece na saída.
- **Sem LLM** (Secrets ausentes): chamamos `buscar_rota` diretamente, nos mesmos três cenários.

In [ ]:
from src.agent.backend_projeto import ProjetoBackend
from src.agent.tools import FlightTools

tools = FlightTools(ProjetoBackend(network, best_pipeline, aircraft_type="Narrow-body"))

if globals().get("TEM_LLM"):
    from src.agent.agent import FlightAgent
    agent = FlightAgent(tools)
    pedidos = [
        "Qual a melhor rota de ABQ para STL?",
        "Tem tempestade no trecho entre DEN e ORD. Qual a melhor rota de ABQ para STL?",
        "Vou de CLT para LAX, mas há tempestade em Dallas. Evite passar por lá.",
    ]
    for p in pedidos:
        r = agent.run(p)
        print("Você:", p)
        print("Agente:", r["resposta"] or f"(sem resposta: {r['erro']})")
        print(f"[{r['latencia_s']}s | {r['tokens_in']}+{r['tokens_out']} tokens | "
              f"ferramentas: {[c['nome'] for c in r['chamadas']]}]\n")
else:
    print("Sem LLM: chamando a ferramenta buscar_rota diretamente.\n")
    cenarios = [
        ("ABQ -> STL", dict(origem="ABQ", destino="STL")),
        ("ABQ -> STL com tempestade em DEN->ORD",
         dict(origem="ABQ", destino="STL",
              eventos=[{"de": "DEN", "para": "ORD", "tipo": "Storm"}])),
        ("CLT -> LAX evitando DFW", dict(origem="CLT", destino="LAX", evitar=["DFW"])),
    ]
    for titulo, args in cenarios:
        r = tools.buscar_rota(**args)
        print(titulo)
        if not r["ok"]:
            print("  erro:", r["erro"], "\n")
            continue
        print("  rota:", " -> ".join(r["rota"]),
              f"| {r['tempo_previsto_min']} min | {r['distancia_km']} km")
        curta = r["rota_mais_curta"]
        if curta:
            print("  mais curta em km:", " -> ".join(curta["rota"]),
                  f"| {curta['tempo_previsto_min']} min | {curta['distancia_km']} km")
            print("  economia:", r["economia_min"], "min\n")

## 6. Avaliação dos prompts

Resultados salvos de uma bateria de 16 pedidos com a versão v1 e a versão v3 do prompt do sistema (arquivos em `results/`). A versão ativa no código é a v1 (`ACTIVE_VERSION` em `src/agent/prompts.py`).

**Análise:** a v3 acrescenta regras (validar códigos desconhecidos e listar todos os aeroportos da rota), mas acertou menos que a v1 e ficou mais lenta. **PREENCHER: uma ou duas frases explicando por que as regras extras não melhoraram o agente e por que a equipe escolheu a v1.**

In [ ]:
import pandas as pd
for v in ("v1", "v3"):
    d = pd.read_csv(f"results/fase3_avaliacao_{v}.csv")
    print(v, f"{int(d.acerto.sum())}/{len(d)} acertos | latência média {d.latencia_s.mean():.1f}s")

## Equipe

- Sofia de Carvalho Brito - 2512130063 - Fase 1 (Formulação e Busca)
- Victor Sousa Pereira - 2512130015 - Fase 2 (Dados e Aprendizado)
- Pedro de Faria Mello - 2512130037 - Fase 3 (Agente com LLM e IA Responsável)